(sec:loc_prop)=
# Localized properties

(sec:esp)=
## ESP charges

Since there is no unique definition for partial charges and no corresponding physical observable, they can be assigned in several ways, such as being derived from the quantum mechanical electrostatic potential

\begin{equation*}
V(\boldsymbol{r}) = 
\sum_{I}
\frac{Z_I e}{4\pi\varepsilon_0 |\boldsymbol{r}-\mathrm{\textbf{R}}_I|} - e
\sum_{\mu,\nu}
D_{\mu\nu}
\int 
\frac{
\phi_\mu^*(\boldsymbol{r}')\phi_\nu(\boldsymbol{r}')
}{
4\pi\varepsilon_0
|\boldsymbol{r}-\boldsymbol{r}'|
}
d^3\boldsymbol{r}'
\end{equation*}

that can be replaced with a potential caused by the partial charges:

\begin{equation*}
\widetilde{V}(\boldsymbol{r}) = 
\sum_{I}
\frac{
q_I
}{
4\pi\varepsilon_0
|\boldsymbol{r}-\textbf{R}_I|
}
\end{equation*}

The Merz–Kollman scheme minimizes the squared norm difference between these two quantities evaluated on a set of grid points in the solvent-accessible region of the molecule with respect to variations in the partial charges and a constraint of a conservation of the total molecular charge – the grid points are distributed on successive layers of scaled van der Waals surfaces. This measure is referred to as the figure-of-merit

\begin{equation*}
\chi_{\mathrm{esp}}^2 = \sum_a \bigl(V(\boldsymbol{r}_a) - \widetilde{V}(\boldsymbol{r}_a)\bigl)^2
\end{equation*}

The resulting electrostatic potential (ESP) charges are obtained by solving the equation

\begin{equation*}
\mathrm{\textbf{A}} \, \mathrm{\textbf{q}} = \mathrm{\textbf{b}}
\end{equation*}

where

\begin{equation*}
A_{JI} =
\frac{1}{4\pi\varepsilon_0}
\sum_{a} \frac{1}{r_{aI}r_{aJ}}
\end{equation*}

and

\begin{equation*}
b_J = \sum_{a} \frac{V_a}{r_{aJ}}
\end{equation*}

**Python script**

In [ ]:
import veloxchem as vlx

xyz_str = """6
Methanol
H      1.2001      0.0363      0.8431
C      0.7031      0.0083     -0.1305
H      0.9877      0.8943     -0.7114
H      1.0155     -0.8918     -0.6742
O     -0.6582     -0.0067      0.1730
H     -1.1326     -0.0311     -0.6482
"""

molecule = vlx.Molecule.read_xyz_string(xyz_str)
basis = vlx.MolecularBasis.read(molecule, "6-31G*")

esp_drv = vlx.RespChargesDriver()
esp_drv.equal_charges = "1=3, 1=4"
esp_charges = esp_drv.compute(molecule, basis, "esp")

Download a {download}`Python script <../input_files/h2o-esp.py>` type of input file to calculate the ESP charges for the water molecule at the HF/6-31G level of theory.

**Text file**

```
@jobs
task: esp charges
@end

@method settings
basis: 6-31G*
@end

@molecule
charge: 0
multiplicity: 1
xyz:  
...
@end
```

Download a {download}`text file <../input_files/h2o-esp.inp>` type of input file to calculate the ESP charges for the water molecule at the HF/6-31G level of theory.

In both cases, the user can control the number of layers of the molecular surface as well as the surface grid point density in these layers (in units of Å$^{-2}$). In the above examples, the recommended default values are employed.

(sec:resp)=
## RESP charges

The restrained electrostatic potential (RESP) charge model is an improvement to the Merz–Kollman scheme as the figure-of-merit $\chi^2_\mathrm{esp}$, is rather insensitive to variations in charges of atoms buried inside the molecule, as illustrated below for methanol and its buried carbon atom in red.

```{figure} ../images/chi_square.png
---
name: chi_square
width: 500px
align: center
---
```

To avoid unphysically large charges of interior atoms, a hyperbolic penalty function is added

\begin{equation*}
\chi_{\mathrm{rstr}}^2 = \alpha \sum_I \bigl((q_I^2+\beta^2)^{1/2}-\beta\bigl)
\end{equation*}

so that the diagonal matrix elements become equal to

\begin{equation*}
A_{JJ} = 
\frac{1}{4\pi\varepsilon_0}
\sum_{a} \frac{1}{r_{aJ}^2} + \alpha \, (q_J^2+\beta^2)^{-1/2}
\end{equation*}

with a dependency on the partial charge. Consequently, RESP charges are obtained by solving the matrix equation iteratively until the charges and Lagrange multipliers become self-consistent. In addition to that, the RESP charge model allows for the introduction of constraints on charges of equivalent atoms due to symmetry operations or bond rotations.

**Python script**

In [ ]:
import veloxchem as vlx

xyz_str = """6
Methanol
H      1.2001      0.0363      0.8431
C      0.7031      0.0083     -0.1305
H      0.9877      0.8943     -0.7114
H      1.0155     -0.8918     -0.6742
O     -0.6582     -0.0067      0.1730
H     -1.1326     -0.0311     -0.6482
"""

molecule = vlx.Molecule.read_xyz_string(xyz_str)
basis = vlx.MolecularBasis.read(molecule, "6-31G*")

resp_drv = vlx.RespChargesDriver()
resp_drv.equal_charges = "1=3, 1=4"
resp_charges = resp_drv.compute(molecule, basis, "resp")

Download a {download}`Python script <../input_files/h2o-resp.py>` type of input file to calculate the RESP charges for the water molecule at the HF/6-31G* level of theory.

**Text file**

```
@jobs
task: resp charges
@end

@method settings
basis: 6-31g*
@end

@resp charges
equal charges: 2 = 3    ! with reference to the atom ordering below
@end

@molecule
charge: 0
multiplicity: 1
xyz:  
...
@end 
```

Download a {download}`text file <../input_files/h2o-resp.inp>` type of input file to calculate the RESP charges for the water molecule at the HF/6-31G* level of theory.

(subsec:bol-weighted-resp)=
### Boltzmann-weighted RESP charges

If is also possible to calculate the Boltzmann-weighted RESP charges for a set of conformers.
In this example, we create three conformers of propanol:

**Python script**

In [11]:
import veloxchem as vlx

xyz_str = """12
propanol
C              0.287122604482        -0.658439248096         1.352543198101
C              0.224057122961        -0.230177036308        -0.123498537286
C             -1.211810163746        -0.086289617161        -0.646362355309
O             -1.865244099222         0.943810244897         0.043408964407
H             -0.138850281128         0.127254486586         2.013590563967
H              1.345443988487        -0.818617209367         1.650405684477
H             -0.266954871175        -1.608743753962         1.513003144654
H              0.758475938136         0.738103480182        -0.253354377301
H              0.752941954799        -0.990737769186        -0.740193839208
H             -1.190106706453         0.157165363176        -1.734264017680
H             -1.766243401522        -1.044101824780        -0.510744786230
H             -1.823078650351         0.710040930866         1.007688030780
"""

molecule = vlx.Molecule.read_xyz_string(xyz_str)
molecule.show(atom_indices=True, width=600, height=450)

conf = vlx.ConformerGenerator()
conformers = conf.generate(molecule)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                                                   
                

In [12]:
conf.show_conformers(number=3, atom_indices=True)

Conformer 1 with energy 14.609 kJ/mol


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Conformer 2 with energy 17.583 kJ/mol


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Conformer 3 with energy 17.765 kJ/mol


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [13]:
# Now we will compute the Boltzmann-weighted RESP charges:
empty_mol = vlx.Molecule()
empty_bas = vlx.MolecularBasis()

resp_settings = { 
    'molecules': conformers['molecules'],
    'filename': 'resp'
}
method_settings = {'basis': '6-31g*'}

resp_drv = vlx.RespChargesDriver()
resp_drv.update_settings(resp_settings, method_settings)
resp_charges = resp_drv.compute(empty_mol, empty_bas, 'resp')

print("Boltzmann-weighted RESP charges =" , resp_charges)


* Info * Found 3 conformers from molecule list.                                                                           
                                                                                                                          
* Info * Processing conformer 1...                                                                                        
                                                                                                                          
                                              Molecular Geometry (Angstroms)                                              
                                                                                                                          
                          Atom         Coordinate X          Coordinate Y          Coordinate Z                           
                                                                                                                          
                

Download a {download}`Python script <../input_files/resp_boltzmann-weighted.py>` type of input file to calculate the Boltzmann-weighted RESP charges for the three conformers of propanol at the HF/6-31G* level of theory.

Note that the argument `'filename': 'resp'` instructs the creation of a folder named `resp_files` which will contain the SCF output of all conformers.

Alternatively, one can also provide a file containing the XYZ coordinates of all conformers:

```python
resp_settings = { 
    'xyz_file': 'all_conformers.xyz'
}
```

**Text file**

```
@jobs
task: resp charges
@end

@method settings
basis: 6-31g*
@end

@resp charges
xyz_file: all_conformers.xyz
@end

@molecule
charge: 0
multiplicity: 1
@end
```

Download a {download}`text file <../input_files/resp_boltzmann-weighted.inp>` type of input file to calculate the Boltzmann-weighted RESP charges for the three conformers of propanol at the HF/6-31G* level of theory, and use the {download}`all_conformers.xyz <../input_files/all_conformers.xyz>` file.

## CHELPG charges

Different choices of grid points in the Merz–Kollman (MK) scheme can be made. CHELPG charges are obtained with grid points chosen on a dense cubic grid with exclusion made of grid points inside the van der Waals molecular volume.

In contrast to the original MK scheme, the calculation of CHELPG charges involve grid points directly outside the van der Waals molecular volume, and since the electrostatic potential is here large, these points will be important for the minimization of the Lagrangian. We note that there is no universal grid-point choice that can be considered best for all situations.

**Python script**

In [ ]:
import veloxchem as vlx

xyz_str = """6
Methanol
H      1.2001      0.0363      0.8431
C      0.7031      0.0083     -0.1305
H      0.9877      0.8943     -0.7114
H      1.0155     -0.8918     -0.6742
O     -0.6582     -0.0067      0.1730
H     -1.1326     -0.0311     -0.6482
"""

molecule = vlx.Molecule.read_xyz_string(xyz_str)
basis = vlx.MolecularBasis.read(molecule, "6-31G*")

esp_drv = vlx.RespChargesDriver()

esp_drv.grid_type = "chelpg"

esp_drv.equal_charges = "1=3, 1=4"

chelpg_charges = esp_drv.compute(molecule, basis, "esp")

## Charge comparison

The localized charges for methanol in the examples above become:

In [ ]:
print("Atom      ESP charge        RESP charge    CHELPG charge")

print(56 * "-")

for label, esp_charge, resp_charge, chelpg_charge in zip(
    molecule.get_labels(), esp_charges, resp_charges, chelpg_charges
):

    print(
        f"{label :s} {esp_charge : 18.6f}{resp_charge : 18.6f}{chelpg_charge : 18.6f}"
    )

print(56 * "-")

print(
    f"Total: {esp_charges.sum() : 13.6f}{resp_charges.sum() : 18.6f}{chelpg_charges.sum() : 18.6f}"
)

(sec:loprop)=
## LoProp charges and polarizabilities

The LoProp approach {cite}`Gagliardi2004` is implemented for the determination of localized (atomic) charges and polarizabilities that enter into polarizable embedding calculations of optical spectra.

**Python script**

In [ ]:
import veloxchem as vlx

molecule = vlx.Molecule.read_molecule_string(
    """
O    0.0000000    0.0000000   -0.1653507
H    0.7493682    0.0000000    0.4424329
H   -0.7493682    0.0000000    0.4424329
"""
)

basis = vlx.MolecularBasis.read(molecule, "ANO-S-VDZP")

scf_drv = vlx.ScfRestrictedDriver()
scf_results = scf_drv.compute(molecule, basis)

loprop_drv = vlx.PEForceFieldGenerator()
loprop_results = loprop_drv.compute(molecule, basis, scf_results)

This calculation gives the following results.

In [ ]:
print("LoProp charges (a.u.):")
print(f"O: {loprop_results['localized_charges'][0] : .4f}")
print(f"H: {loprop_results['localized_charges'][1] : .4f}")
print(f"H: {loprop_results['localized_charges'][2] : .4f}")

print("\nLoProp polarizabilities (a.u.):")
print("     xx     yy     zz")
print(
    f"O: {loprop_results['localized_polarizabilities'][0][0]:5.2f}{loprop_results['localized_polarizabilities'][0][3]:7.2f}{loprop_results['localized_polarizabilities'][0][5]:7.2f}"
)
print(
    f"H: {loprop_results['localized_polarizabilities'][1][0]:5.2f}{loprop_results['localized_polarizabilities'][1][3]:7.2f}{loprop_results['localized_polarizabilities'][1][5]:7.2f}"
)
print(
    f"H: {loprop_results['localized_polarizabilities'][2][0]:5.2f}{loprop_results['localized_polarizabilities'][2][3]:7.2f}{loprop_results['localized_polarizabilities'][2][5]:7.2f}"
)

Download a {download}`Python script <../input_files/h2o-loprop.py>` type of input file to calculate the LOPROP charges and atomic polarizabilities for the water molecule at the B3LYP/ANO-S-VDPZ level of theory.

**Text file**
````
@jobs
task: loprop
@end

@method settings
xcfun: b3lyp
basis: ANO-S-VDZP ! An ANO type of basis set should be used
@end

@molecule
charge: 0
multiplicity: 1
xyz:
...
@end
````

Download a {download}`text file <../input_files/h2o-loprop.inp>` the input file to calculate the LoProp charges and atomic polarizabilities for the water molecule at the B3LYP/ANO-S-VDPZ level of theory.

```{image} ../images/water.png
:alt: cover
:class: bg-primary mb-1
:width: 300px
:align: center
```